# CafeBERT exp027 - đánh giá ABSA thực tế (Shopee)

**Mô hình:** `cafebert/lora/exp027` = base `uitnlp/CafeBERT` (HuggingFace) + LoRA + head 7 khía cạnh; id serving `cafebert_exp027`.

**Base model:** lấy từ HuggingFace qua `checkpoint` trong config đã hợp nhất (`uitnlp/CafeBERT`), đúng như lượt huấn luyện. `EncoderRuntime` tự `snapshot_download` nếu chưa có trong HF cache (`local_files_only = False`); không cần thư mục `cafebert-base` trên Drive.

**Checkpoint huấn luyện (riêng exp027):** `experiments/cafebert/lora/exp027/results/43e40f61/model/best` (tức `<results_root>/cafebert/lora/exp027/results/43e40f61/model/best`) gồm LoRA adapter + head. Đây là phần **thêm** so với base HuggingFace; thiếu file nào thì notebook dừng và in đường dẫn + file còn thiếu.

**Chế độ:** chỉ suy luận - **không huấn luyện**, không cập nhật trọng số, không dùng để chọn mô hình.

**Cấu hình:** nạp từ mã nguồn bằng `experiments.load("cafebert", "lora", "exp027")`. Không đọc `experiments/.../config.yaml` trên Drive; `configs/serving.yaml` chỉ dùng cho labels/aspects/default shell, không phải nguồn sự thật.

**Dữ liệu:** `data/raw/realworld/v0.1.0/shoppe_absa.csv` - 60 đánh giá son môi Shopee. Cột `text` được đưa **nguyên văn** vào serving runtime; không clean/normalize/segment bên ngoài runtime.

**Cần tải lên Drive:** chỉ checkpoint adapter+head của exp027. Không cần tải `cafebert-base`; base lấy từ HuggingFace.


In [ ]:
# --- BẢN CODE ĐÃ GHIM ---
REPO_URL = "https://github.com/TrieuKhac-dev/SentimentX"
REPO_BRANCH = "main"
REPO_SHA = "05906f647f7364636665236a773b9954e4b92ee5"

import importlib
import os
import pathlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def repo_root():
    if IN_COLAB:
        return pathlib.Path("/content/SentimentX")
    here = pathlib.Path.cwd()
    for candidate in [here] + list(here.parents):
        if (candidate / "configs" / "paths.yaml").is_file() and (candidate / "src" / "__init__.py").is_file():
            return candidate
    return here


def git(*args):
    done = subprocess.run(["git"] + [str(item) for item in args], capture_output=True, text=True)
    print("  git {} -> {}".format(" ".join(str(item) for item in args), done.returncode))
    if done.returncode != 0:
        print("    " + ((done.stdout or "") + (done.stderr or "")).strip()[-400:])
    return done.returncode


def head_sha(directory):
    done = subprocess.run(["git", "-C", str(directory), "rev-parse", "HEAD"],
                          capture_output=True, text=True)
    return done.stdout.strip() if done.returncode == 0 else ""


CODE_DIR = repo_root()

if IN_COLAB:
    from google.colab import drive
    if not pathlib.Path("/content/drive").is_mount():
        drive.mount("/content/drive")
    has_source = (CODE_DIR / "configs" / "paths.yaml").is_file() and (CODE_DIR / "src" / "__init__.py").is_file()
    if (not has_source) or head_sha(CODE_DIR) != REPO_SHA:
        if CODE_DIR.exists() and any(CODE_DIR.iterdir()) and not (CODE_DIR / ".git").is_dir():
            raise SystemExit("DỪNG: {} đã có nhưng không phải git repo. Xoá rồi chạy lại.".format(CODE_DIR))
        if not (CODE_DIR / ".git").is_dir():
            CODE_DIR.mkdir(parents=True, exist_ok=True)
            if git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(CODE_DIR)) != 0:
                raise SystemExit("DỪNG: không clone được {}".format(REPO_URL))
        git("-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", REPO_SHA)
        if git("-C", str(CODE_DIR), "checkout", "--detach", REPO_SHA) != 0:
            raise SystemExit("DỪNG: không checkout được commit {}".format(REPO_SHA))
    os.environ["HF_HOME"] = "/content/hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
    pathlib.Path("/content/hf_cache").mkdir(parents=True, exist_ok=True)
else:
    if not (CODE_DIR / "configs" / "paths.yaml").is_file():
        raise SystemExit("DỪNG: không tìm thấy gốc repo từ {}".format(pathlib.Path.cwd()))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
sys.path_importer_cache.pop(str(CODE_DIR), None)

from src.api import bootstrap, paths, runtime

info = bootstrap.prepare()
print("prepare(): colab={} drive={} stop={}".format(info["colab"], info["drive"], info["stop"]))
if IN_COLAB:
    drive_root = info["drive"] or pathlib.Path("/content/drive/MyDrive/SentimentX")
    os.environ["SENTIMENTX_DATA_ROOT"] = str(drive_root / "data")
    os.environ["SENTIMENTX_RESULTS_ROOT"] = str(drive_root / "experiments")
    print("Drive root :", drive_root)
else:
    os.environ.pop("SENTIMENTX_DATA_ROOT", None)
    os.environ.pop("SENTIMENTX_RESULTS_ROOT", None)

# Gói suy luận: ghim đúng bộ phiên bản của môi trường dự án; không cài torch.
# Colab có torchao 0.10.0 mà peft 0.21.0 từ chối khi bọc LoRA, nên gỡ torchao sau khi cài.
PACKAGES = [
    "transformers==5.17.0",
    "huggingface_hub==1.31.0",
    "peft==0.21.0",
    "accelerate==1.15.0",
    "tokenizers==0.23.2",
    "safetensors==0.8.0",
    "sentencepiece==0.2.2",
    "protobuf==6.33.6",
    "PyYAML",
    "pandas",
]
if IN_COLAB:
    print("Cài gói suy luận:", ", ".join(PACKAGES))
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + PACKAGES,
        capture_output=True, text=True)
    print("  pip install ->", done.returncode)
    if done.returncode != 0:
        print(((done.stdout or "") + (done.stderr or "")).strip()[-1200:])
        raise SystemExit("DỪNG: cài gói thất bại.")

    print("  đang gỡ torchao (LoRA của dự án không dùng gói này)...")
    gone = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
        capture_output=True, text=True)
    sys.modules.pop("torchao", None)
    print("  pip uninstall torchao ->", gone.returncode)

try:
    import transformers
except Exception as exc:
    print("DỪNG: import transformers thất bại: {}: {}".format(type(exc).__name__, exc))
    print("  Thường do huggingface_hub cũ còn sót. Chạy lại ô này, rồi Runtime > Restart session.")
    raise SystemExit(1)


def module_version(name):
    try:
        module = importlib.import_module(name)
    except Exception as exc:
        return "not installed ({})".format(type(exc).__name__)
    return str(getattr(module, "__version__", "unknown"))


print("Phiên bản gói:")
print("  {:<16} {}".format("python", sys.version.split()[0]))
for label, name in (
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("huggingface_hub", "huggingface_hub"),
    ("peft", "peft"),
    ("accelerate", "accelerate"),
    ("tokenizers", "tokenizers"),
    ("safetensors", "safetensors"),
):
    print("  {:<16} {}".format(label, module_version(name)))

print("CODE_DIR   :", CODE_DIR)
print("DATA_ROOT  :", paths.data_root())
print("RESULTS    :", paths.results_root())

# Kiểm tra tokenizer trước khi nạp config: tải/ghim base tokenizer vào HF cache.
# sentencepiece/protobuf không khớp làm transformers rơi sang TikToken và mọi dòng lỗi.
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained("uitnlp/CafeBERT", local_files_only=False)
_ids = _tok("tôi thích sản phẩm này")["input_ids"]
print("Tokenizer CafeBERT: OK; sample tokens =", len(_ids))
if not _ids:
    raise SystemExit("DỪNG: tokenizer CafeBERT trả về rỗng.")


In [ ]:
# Cấu hình ĐÃ HỢP NHẤT từ mã nguồn (không đọc config.yaml trên Drive).
from dataclasses import replace

from src.api import experiments, paths
from serving.config import load_settings

MODEL_ID = "cafebert"
METHOD = "lora"
EXP_ID = "exp027"
SERVING_ID = "cafebert_exp027"
CHECKPOINT_HASH = "43e40f61"
CHECKPOINT_FILES = ("adapter_model.safetensors", "adapter_config.json", "head.pt", "head_config.json")

merged = experiments.load(MODEL_ID, METHOD, EXP_ID)
config = merged["config"]

if config.get("approach") != "encoder":
    raise SystemExit("DỪNG: exp027 phải có approach='encoder', đang là {!r}.".format(config.get("approach")))

pre = config.get("preprocess") or {}
inf = config.get("inference") or {}
if "max_length" not in pre or "segmenter" not in pre:
    raise SystemExit("DỪNG: merged config thiếu preprocess.max_length/segmenter.")
if "dtype" not in inf:
    raise SystemExit("DỪNG: merged config thiếu inference.dtype.")

checkpoint = paths.results_root() / MODEL_ID / METHOD / EXP_ID / "results" / CHECKPOINT_HASH / "model" / "best"
missing = [name for name in CHECKPOINT_FILES if not (checkpoint / name).is_file()]
if missing:
    print("DỪNG: không thấy checkpoint CafeBERT exp027.")
    print("  đường dẫn :", checkpoint)
    print("  còn thiếu :", ", ".join(missing))
    print("  cần adapter_model.safetensors, adapter_config.json, head.pt, head_config.json trong đúng thư mục trên.")
    raise SystemExit(1)

base = str(config.get("checkpoint") or "")
if not base:
    raise SystemExit("DỪNG: merged config không có checkpoint (HF id) làm base cho CafeBERT.")

settings = load_settings(models_override=[SERVING_ID])
shell = settings.models[SERVING_ID]
runtime_values = dict(shell.runtime)
runtime_values.update({
    "checkpoint": str(checkpoint),
    "base": base,
    "local_files_only": False,
    "max_length": int(pre["max_length"]),
    "segmenter": str(pre["segmenter"]),
    "dtype": str(inf["dtype"]),
})
settings.models[SERVING_ID] = replace(shell, runtime=runtime_values)
spec = settings.models[SERVING_ID]

print("Thí nghiệm :", "{}/{}/{}".format(MODEL_ID, METHOD, EXP_ID))
print("Đường chạy :", config.get("approach"))
print("Base       :", base, "(merged config checkpoint; HuggingFace)")
print("Adapter    :", checkpoint, "(hash {})".format(CHECKPOINT_HASH))
print("max_length :", runtime_values["max_length"], "| segmenter:", runtime_values["segmenter"], "| dtype:", runtime_values["dtype"])
print("Aspects    :", settings.labels.get("aspects"))
print("Lớp config :")
for label, path in merged["layers"]:
    print("  {:<12} {}".format(label, path))


In [ ]:
# Dữ liệu thô + suy luận bằng serving runtime (đúng đường serving, không gọi encoder_run).
import hashlib
import time
from datetime import datetime, timezone

import pandas as pd

from src.api import paths, runtime as workflow_runtime
from serving.inference.encoder import EncoderRuntime
from src.evaluation import scorers

ASPECTS = [str(name) for name in settings.labels["aspects"]]
TASK = {"label_space": "binary", "neutral_policy": "drop", "not_mentioned": "separate"}
LABELS = {0: "not_mentioned", 1: "positive", 2: "negative"}
CHUNK = 16
INCLUDE_PROBS = True

data_path = paths.data_root() / "raw" / "realworld" / "v0.1.0" / "shoppe_absa.csv"
if not data_path.is_file():
    data_path = paths.root() / "data" / "raw" / "realworld" / "v0.1.0" / "shoppe_absa.csv"
if not data_path.is_file():
    raise SystemExit("DỪNG: không thấy data/raw/realworld/v0.1.0/shoppe_absa.csv ở data root hoặc repo.")

frame = pd.read_csv(data_path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
if len(frame) != 60:
    raise SystemExit("DỪNG: shoppe_absa.csv phải có 60 dòng, đang có {}.".format(len(frame)))
missing_cols = [name for name in ["text"] + ASPECTS if name not in frame.columns]
if missing_cols:
    raise SystemExit("DỪNG: CSV thiếu cột: {}".format(", ".join(missing_cols)))

texts = frame["text"].tolist()
gold_rows = []
for index in range(len(frame)):
    row = {}
    for aspect in ASPECTS:
        value = str(frame.iloc[index][aspect]).strip()
        if value not in ("0", "1", "2"):
            raise SystemExit("DỪNG: nhãn {!r} của cột {} dòng {} không thuộc 0/1/2.".format(value, aspect, index))
        row[aspect] = int(value)
    gold_rows.append(row)

row_ids = frame["row_id"].tolist() if "row_id" in frame.columns else list(range(len(frame)))
urls = frame["url"].tolist() if "url" in frame.columns else [""] * len(frame)

csv_sha = hashlib.sha256(data_path.read_bytes()).hexdigest()
run_id = hashlib.sha256((str(spec.version) + csv_sha).encode("utf-8")).hexdigest()[:8]
out_dir = paths.results_root() / "realworld" / SERVING_ID / run_id
out_dir.mkdir(parents=True, exist_ok=True)


def predict_chunked(runtime_obj, texts_in, chunk, include_probs):
    predictions, errors = [], []
    for start in range(0, len(texts_in), chunk):
        batch = texts_in[start:start + chunk]
        try:
            results = runtime_obj.predict(batch, include_probs=include_probs)
            if not isinstance(results, list) or len(results) != len(batch):
                raise RuntimeError(
                    "số kết quả {} khác số dòng {}".format(
                        len(results) if isinstance(results, list) else "?", len(batch)))
            predictions.extend(results)
            continue
        except Exception as exc:
            print("  khối {}-{} lỗi ({}); thử lại từng dòng.".format(
                start, start + len(batch) - 1, exc))
        for offset, text in enumerate(batch):
            index = start + offset
            try:
                one = runtime_obj.predict([text], include_probs=include_probs)
                if not one:
                    raise RuntimeError("không có kết quả")
                predictions.append(one[0])
            except Exception as row_exc:
                predictions.append(None)
                errors.append({
                    "index": index,
                    "row_id": row_ids[index],
                    "error": "{}: {}".format(type(row_exc).__name__, row_exc),
                })
    return predictions, errors


started_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
started = time.perf_counter()
runtime_obj = EncoderRuntime()
try:
    runtime_obj.load(spec, settings)
    predictions, errors = predict_chunked(runtime_obj, texts, CHUNK, INCLUDE_PROBS)
finally:
    runtime_obj.unload()
elapsed = round(time.perf_counter() - started, 3)

if errors and len(errors) == len(texts):
    print("DỪNG: toàn bộ {} dòng lỗi runtime; không chấm điểm.".format(len(texts)))
    for item in errors[:5]:
        print("  row", item.get("row_id"), "-", item.get("error"))
    raise SystemExit(1)
if errors:
    print("CẢNH BÁO: {} dòng lỗi runtime; vẫn chấm điểm trên {} dòng còn lại.".format(
        len(errors), len(texts) - len(errors)))
    for item in errors[:3]:
        print("  row", item.get("row_id"), "-", item.get("error"))

pred_rows = []
for prediction in predictions:
    if prediction is None:
        pred_rows.append(None)
        continue
    pred_rows.append({
        aspect: (None if prediction.get(aspect) is None else prediction[aspect].code)
        for aspect in ASPECTS
    })

print("Đã dự đoán {} dòng | lỗi runtime: {}".format(len(predictions), len(errors)))
print("Hash8      :", run_id)
print("Thư mục    :", out_dir)
print("Dữ liệu    :", data_path)


In [ ]:
# Chấm điểm bằng src.evaluation.scorers.write + ghi predictions.csv, run_meta.json.
import json
from datetime import datetime, timezone

import pandas as pd

from src.evaluation import scorers

prediction_columns = {
    "row_id": row_ids,
    "text": texts,
    "url": urls,
}
for aspect in ASPECTS:
    prediction_columns["gold_" + aspect] = [row[aspect] for row in gold_rows]
    prediction_columns["pred_" + aspect] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].code
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_label"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].label
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_confidence"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].confidence
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_quote"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].quote
        for prediction in predictions
    ]
    prediction_columns["pred_" + aspect + "_reason"] = [
        None if prediction is None or prediction.get(aspect) is None else prediction[aspect].reason
        for prediction in predictions
    ]

predictions_path = out_dir / "predictions.csv"
pd.DataFrame(prediction_columns).to_csv(predictions_path, index=False, encoding="utf-8-sig")

run_info = {
    "model": SERVING_ID,
    "model_version": spec.version,
    "run_id": run_id,
    "dataset": "shoppe_absa",
    "data_path": str(data_path),
    "csv_sha256": csv_sha,
    "aspects": ASPECTS,
    "task": TASK,
    "labels": LABELS,
    "chunk": CHUNK,
    "include_probs": INCLUDE_PROBS,
    "n_rows": len(texts),
    "n_errors": len(errors),
    "runtime_errors": errors,
    "config_dir": str(merged.get("dir")),
    "config_layers": list(merged.get("layers") or []),
    "config_sources": {
        key: merged.get("sources", {}).get(key)
        for key in ("approach", "checkpoint", "preprocess.max_length", "preprocess.segmenter",
                    "inference.dtype", "inference.quantization", "data.dataset", "data.version")
        if merged.get("sources", {}).get(key)
    },
    "runtime_spec": {
        key: spec.runtime.get(key)
        for key in ("checkpoint", "base", "base_revision", "max_length", "segmenter", "dtype",
                    "model_dir", "quantization", "max_new_tokens", "prompt", "examples", "system",
                    "local_files_only")
        if spec.runtime.get(key) is not None
    },
    "env": workflow_runtime.env_name(),
    "data_root": str(paths.data_root()),
    "results_root": str(paths.results_root()),
    "started_at": started_at,
    "seconds": elapsed,
}

samples = scorers.Samples.build(
    ASPECTS,
    gold_rows,
    pred_rows,
    task=dict(TASK),
    labels=dict(LABELS),
    sample_ids=texts,
    meta=dict(run_info),
)

names = list(config.get("scores") or scorers.available())
paper = list(config.get("scores_paper") or [])
save = config.get("save") or {}
written = scorers.write(
    out_dir,
    samples,
    names=names,
    paper=paper or None,
    save_confusion=bool(save.get("confusion", True)),
    save_plots=False,
    extra=dict(run_info),
)

metrics_path = out_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
scores = metrics.get("scores") or {}
accuracy = scores.get("accuracy") or {}
detection = scores.get("aspect_detection") or {}
accuracy_macro = accuracy.get("macro")
accuracy_micro = accuracy.get("micro")
detection_macro = (detection.get("macro") or {}).get("f1")
detection_micro = (detection.get("micro") or {}).get("f1")

files = {name: str(path) for name, path in written.items()}
files["predictions.csv"] = str(predictions_path)
run_meta = dict(run_info)
run_meta.update({
    "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "out_dir": str(out_dir),
    "files": files,
    "metrics": {
        "accuracy_macro": accuracy_macro,
        "accuracy_micro": accuracy_micro,
        "accuracy_cells": accuracy.get("cells"),
        "detection_f1_macro": detection_macro,
        "detection_f1_micro": detection_micro,
    },
})
run_meta_path = out_dir / "run_meta.json"
run_meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

print("=" * 72)
print("Mô hình       :", SERVING_ID)
print("Phiên bản     :", spec.version)
print("Hash8         :", run_id)
print("Thư mục       :", out_dir)
print("Accuracy      : macro={} | micro={} | số ô={}".format(
    accuracy_macro, accuracy_micro, accuracy.get("cells")))
print("Detection F1  : macro={} | micro={}".format(detection_macro, detection_micro))
print("Lỗi runtime   :", len(errors))
print("Thời gian     :", elapsed, "giây")
print("File          :")
for name in ("predictions.csv", "metrics.json", "metrics.csv", "mispredictions.csv", "run_meta.json"):
    print("  {} -> {}".format(name, out_dir / name))
print("=" * 72)
